---
layout: post
codemirror: true
title: 3.15 Random Values HW
categories: [Python]
lesson_language: Python
lesson_topic: Random-Values HW
lesson_part: interactive
lesson_type: lesson
permalink: /python/random-hw
author: kelerviafang-ux
---

## SFI Backend QA Simulator

Click **Run** (or press **Ctrl+Enter / Cmd+Enter**) to test randomly selected part records and backend actions in the browser.

This simulator uses an **in-memory backend**. Each run starts with `existing_spec_numbers`, then successful creation, updates, and deletion change that backend for later tests. It does not send requests to the automotive backend.

- **GET/search:** `200 OK` when a spec exists; otherwise `404 Not Found`.
- **POST/create:** `201 Created` for a new spec; `409 Conflict` rejects duplicates without overwriting the existing record.
- **PUT/update:** `200 OK` updates an existing record; otherwise `404 Not Found`.
- **DELETE/remove:** `200 OK` removes an existing record; otherwise `404 Not Found`.

Edit `parts`, `actions`, `existing_spec_numbers`, or `test_count` to reuse the simulator. The example uses `seed=4` to reproduce 12 randomized tests that cover all four actions, a new spec, and a later duplicate creation attempt. Change the seed to explore another sequence, or use `seed=None` for a fresh random sequence each time.

Every test prints the selected structured record, action, and HTTP-style result. `run_qa_simulator()` also returns the test results and final backend records for further analysis. Your original input records and spec-number list stay unchanged.


In [ ]:
# CODE_RUNNER: Run randomized SFI backend QA and inspect each selected record, action, and result. | python_runtime: browser

import random

# These are sample QA records, not a live SFI specification catalog.
parts = [
    {
        "product_name": "Replacement Flywheels",
        "category": "Auto Racing",
        "spec_number": "1.1",
        "effective_date": "2026-10-01"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2",
        "effective_date": "2026-10-02"
    },
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1",
        "effective_date": "2026-10-03"
    }
]

actions = ["GET/search", "POST/create", "PUT/update", "DELETE/remove"]
existing_spec_numbers = ["1.1", "2.1"]
test_count = 12


def simulate_backend_action(record, action, backend):
    """Apply one action to an in-memory backend keyed by spec number.

    Args:
        record (dict): Part with a spec_number and other SFI part fields.
        action (str): GET/search, POST/create, PUT/update, or DELETE/remove.
        backend (dict): Mutable spec-to-record mapping shared between tests.

    Returns:
        dict: status_code, message, and data (a record copy or None).
            Successful create/update/delete changes backend. Duplicate create
            returns 409 and preserves the stored record. Unknown actions raise
            ValueError.
    """
    spec = record["spec_number"]
    exists = spec in backend

    # Check duplicates before assignment so POST cannot overwrite an old record.
    if action == "POST/create":
        if exists:
            return {"status_code": 409, "message": "Conflict: duplicate spec " + spec, "data": None}
        backend[spec] = record.copy()
        return {"status_code": 201, "message": "Created spec " + spec, "data": backend[spec].copy()}

    if action not in ("GET/search", "PUT/update", "DELETE/remove"):
        raise ValueError("Unsupported backend action: " + str(action))
    if not exists:
        return {"status_code": 404, "message": "Not Found: spec " + spec, "data": None}

    if action == "GET/search":
        return {"status_code": 200, "message": "OK: found spec " + spec, "data": backend[spec].copy()}
    if action == "PUT/update":
        backend[spec] = record.copy()
        return {"status_code": 200, "message": "OK: updated spec " + spec, "data": backend[spec].copy()}

    removed = backend.pop(spec)
    return {"status_code": 200, "message": "OK: removed spec " + spec, "data": removed.copy()}


def run_qa_simulator(part_records, backend_actions, existing_specs, test_count=5, seed=None):
    """Print repeated random QA tests and return their structured results.

    Args:
        part_records (list[dict]): Nonempty candidates with product_name,
            category, spec_number, and effective_date string fields.
        backend_actions (list[str]): Nonempty selection of supported actions.
        existing_specs (collection[str]): Initially stored spec numbers. Those
            found in part_records use that record; others start as spec-only
            placeholders because their full backend details were not supplied.
        test_count (int): Nonnegative number of independent random selections.
        seed (int or None): Reproduce a sequence, or None for fresh randomness.

    Returns:
        dict: results (list of selected records, actions, and responses) and
            backend (final spec-to-record mapping). Each call starts fresh and
            copies the inputs. Backend state persists between tests in a call.
    """
    if not part_records or not backend_actions:
        raise ValueError("Provide at least one part record and backend action.")
    if not isinstance(test_count, int) or isinstance(test_count, bool) or test_count < 0:
        raise ValueError("test_count must be a nonnegative integer.")
    supported = {"GET/search", "POST/create", "PUT/update", "DELETE/remove"}
    if any(action not in supported for action in backend_actions):
        raise ValueError("The action list contains an unsupported backend action.")

    # A local generator makes seeded runs repeatable without changing global randomness.
    rng = random.Random(seed)
    catalog = {part["spec_number"]: part.copy() for part in part_records}
    backend = {spec: catalog.get(spec, {"spec_number": spec}).copy() for spec in existing_specs}
    results = []
    print("Starting specs:", sorted(backend))

    for run_number in range(1, test_count + 1):
        # choice() samples with replacement, so the same record can be tested again.
        selected_record = rng.choice(part_records).copy()
        selected_action = rng.choice(backend_actions)
        response = simulate_backend_action(selected_record, selected_action, backend)
        results.append({"run": run_number, "record": selected_record,
                        "action": selected_action, "response": response})

        print(f"\nTest {run_number}/{test_count}")
        print("Selected record:", selected_record)
        print("Selected action:", selected_action)
        print(f"Result: HTTP {response['status_code']} - {response['message']}")

    print("\nCompleted tests:", len(results))
    print("Final specs:", sorted(backend))
    return {"results": results, "backend": backend}


# Reuse this procedure with a different dataset, test count, or random seed.
qa_report = run_qa_simulator(parts, actions, existing_spec_numbers, test_count, seed=4)
